<a href="https://colab.research.google.com/github/chiaraco16/COVID-19-analysis/blob/main/COVID_L4_Data_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 L4 – Data Modeling: Dimensional Fact Model & Star Schema

**Progetto:** COVID-19 Trend Analysis — Chiara Costantino (277081)


**Obiettivo:** Costruire il modello dimensionale per il DW COVID-19 seguendo l'approccio **data-driven**:
1. Caricare il Reconciled DB (output di L1–L3)
2. Scoprire le Dipendenze Funzionali
3. Costruire l'**Attribute Tree**
4. Classificare le misure per additività
5. Derivare il **DFM Fact Schema** e lo **Star Schema** (Fact_Covid_Trend, Dim_Location_Context, Dim_Time)
6. Generare il **DDL MySQL**
7. Produrre il **Glossario delle misure**
8. Esportare `schema.json` per il passo ETL

### Schema concettuale atteso:

```
Fact_Covid_Trend
   ├── Dim_Location_Context  (iso_code → location → continent)
   └── Dim_Time              (date → month → quarter → year)
```


## 0. Connessione a Google Drive e Configurazione Percorsi

In [1]:
# Monto Google Drive per leggere i file del progetto
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Percorsi: input = tabelle pulite di L3, output = file del DW (fatto, dimensioni, schema)
import os

BASE_PATH = '/content/drive/MyDrive/DataWarehouse/'
# Cartella base del progetto su Google Drive

# Input: output di L3 (tabelle pulite)
EPIDEMIC_CLEAN    = BASE_PATH + 'L3_epidemic_trend_clean.csv'
PREVENTION_CLEAN  = BASE_PATH + 'L3_prevention_measures_clean.csv'
LOCATIONS_CLEAN   = BASE_PATH + 'L3_locations_clean.csv'

# Output di questo notebook
RECONCILED_PATH   = BASE_PATH + 'reconciled/'
OUTPUT_SCHEMA     = BASE_PATH + 'schema.json'
OUTPUT_DDL        = BASE_PATH + 'schema_dw.sql'
OUTPUT_GLOSSARY   = BASE_PATH + 'glossary_measures.csv'
OUTPUT_FACT       = BASE_PATH + 'Fact_Covid_Trend.csv'
OUTPUT_DIM_LOC    = BASE_PATH + 'Dim_Location_Context.csv'
OUTPUT_DIM_TIME   = BASE_PATH + 'Dim_Time.csv'

os.makedirs(RECONCILED_PATH, exist_ok=True)
print('Percorsi configurati')
print(f'   Base: {BASE_PATH}')
print(f'   Reconciled: {RECONCILED_PATH}')

Percorsi configurati
   Base: /content/drive/MyDrive/DataWarehouse/
   Reconciled: /content/drive/MyDrive/DataWarehouse/reconciled/


## 1. Librerie

In [3]:
# Installo le librerie usate nel notebook
!pip install pandas numpy networkx matplotlib --quiet

In [4]:
# Importo le librerie del notebook
import pandas as pd
import numpy as np
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from datetime import datetime

# Impostazioni di visualizzazione
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
np.random.seed(42)

print('Librerie caricate')

Librerie caricate


## 2. Caricamento Reconciled DB (output L3)

Carico le tre tabelle pulite prodotte in L3:

| Tabella | Chiave primaria | Descrizione |
|---------|----------------|-------------|
| `Locations` | `iso_code` | Attributi geografici e demografici per paese |
| `Epidemic_Trend` | `(iso_code, date)` | Misure giornaliere di flusso epidemico |
| `Prevention_Measures` | `(iso_code, date)` | Misure cumulative di vaccinazione e politiche |


In [5]:
# Carico le 3 tabelle pulite prodotte dal notebook L3 (il Reconciled DB).
locations_df  = pd.read_csv(LOCATIONS_CLEAN)
epidemic_df   = pd.read_csv(EPIDEMIC_CLEAN, parse_dates=['date'])
prevention_df = pd.read_csv(PREVENTION_CLEAN, parse_dates=['date'])

print(f'\n Locations:           {locations_df.shape[0]:>6,} righe × {locations_df.shape[1]:>2} colonne')
print(f'Epidemic_Trend:      {epidemic_df.shape[0]:>6,} righe × {epidemic_df.shape[1]:>2} colonne')
print(f'Prevention_Measures: {prevention_df.shape[0]:>6,} righe × {prevention_df.shape[1]:>2} colonne')
print(f'\nPaesi unici:  {locations_df["iso_code"].nunique()}')
print(f'Periodo:      {epidemic_df["date"].min().date()} → {epidemic_df["date"].max().date()}')


 Locations:              171 righe ×  9 colonne
Epidemic_Trend:      284,298 righe ×  6 colonne
Prevention_Measures: 284,298 righe ×  5 colonne

Paesi unici:  171
Periodo:      2020-01-01 → 2024-08-14


In [21]:
# Controllo veloce delle colonne delle tre tabelle e anteprima struttura
print('\nColonne Epidemic_Trend:')
print(list(epidemic_df.columns))
print('\nColonne Prevention_Measures:')
print(list(prevention_df.columns))
print('\nColonne Locations:')
print(list(locations_df.columns))


Colonne Epidemic_Trend:
['iso_code', 'date', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths', 'month', 'quarter', 'year']

Colonne Prevention_Measures:
['iso_code', 'date', 'people_vaccinated', 'new_tests', 'stringency_index']

Colonne Locations:
['iso_code', 'continent', 'location', 'population', 'population_density', 'median_age', 'gdp_per_capita', 'hospital_beds_per_thousand', 'human_development_index']


## 3. Scoperta delle Dipendenze Funzionali (FD)

Una dipendenza funzionale **X → Y** significa: *ogni valore di X determina esattamente un valore di Y*.

L'approccio **data-driven** estrae le FD direttamente dai dati del Reconciled DB per poi costruire l'Attribute Tree.


| Determinante X | Dipendente Y | Interpretazione |
|---------------|--------------|-----------------|
| `iso_code` | `location` | Ogni codice ISO identifica un paese univoco |
| `iso_code` | `continent` | Ogni paese appartiene a un solo continente |
| `iso_code` | `population` | La popolazione è fissa per paese |
| `date` | `month` | Ogni data appartiene a un mese |
| `date` | `quarter` | Ogni data appartiene a un trimestre |
| `date` | `year` | Ogni data appartiene a un anno |


In [22]:
# Funzione che scopre le dipendenze funzionali X -> Y
# (FD valida se ogni valore di X corrisponde a un solo valore di Y)
def discover_fd(df,
                determinants,
                dependents,
                min_support=2):
    """
    Scopre le Dipendenze Funzionali X → Y nel DataFrame.

    Args:
        df: DataFrame da analizzare
        determinants: lista di colonne candidate come determinanti (LHS)
        dependents: lista di colonne candidate come dipendenti (RHS)
        min_support: numero minimo di valori distinti in X per considerare la FD

    Returns:
        lista di tuple (X, Y) dove X → Y è verificata nel dataset
    """
    fds = []

    for col_a in determinants:
        if col_a not in df.columns:
            continue
        if df[col_a].nunique() < min_support:
            continue


        for col_b in dependents:
            if col_b == col_a or col_b not in df.columns:
                continue

            # Conto quanti valori distinti di Y esistono per ogni valore di X
            mapping = (df.dropna(subset=[col_a, col_b])
                         .groupby(col_a)[col_b]
                         .nunique())
            # Se ogni valore di X mappa ad esattamente 1 valore di Y → FD valida
            if (mapping <= 1).all():
                card_x = df[col_a].nunique()
                card_y = df[col_b].nunique()
                fds.append({
                    'X': col_a,
                    'Y': col_b,
                    'card_X': card_x,
                    'card_Y': card_y,
                    'ratio': round(card_y / card_x, 4) if card_x > 0 else 0
                })

    return fds


In [8]:
# Cerco le dipendenze funzionali nella tabella Locations (es. iso_code -> location)
# FD su Locations (attributi geografici/demografici)
LOCATION_DETERMINANTS = ['iso_code']
LOCATION_DEPENDENTS   = [c for c in locations_df.columns
                          if c not in ['iso_code'] and locations_df[c].dtype == object]
LOCATION_DEPENDENTS  += ['continent', 'location', 'population']

fds_locations = discover_fd(locations_df,
                             determinants=LOCATION_DETERMINANTS,
                             dependents=[c for c in locations_df.columns if c != 'iso_code'])

print(f'FD scoperte in Locations: {len(fds_locations)}')
fds_loc_df = pd.DataFrame(fds_locations)
print(fds_loc_df[['X','Y','card_X','card_Y']].to_string(index=False))

FD scoperte in Locations: 8
       X                          Y  card_X  card_Y
iso_code                  continent     171       6
iso_code                   location     171     171
iso_code                 population     171     171
iso_code         population_density     171     157
iso_code                 median_age     171     115
iso_code             gdp_per_capita     171     142
iso_code hospital_beds_per_thousand     171      81
iso_code    human_development_index     171      51


In [9]:
# Dipendenze funzionali sulle date: date -> month -> quarter -> year
# FD su date
epidemic_df['month']   = epidemic_df['date'].dt.to_period('M').astype(str)
epidemic_df['quarter'] = epidemic_df['date'].dt.to_period('Q').astype(str)
epidemic_df['year']    = epidemic_df['date'].dt.year

TIME_DETERMINANTS = ['date']
TIME_DEPENDENTS   = ['month', 'quarter', 'year']

fds_time = discover_fd(epidemic_df,
                        determinants=TIME_DETERMINANTS,
                        dependents=TIME_DEPENDENTS)

print(f'\nFD temporali scoperte: {len(fds_time)}')
fds_time_df = pd.DataFrame(fds_time)
print(fds_time_df[['X','Y','card_X','card_Y']].to_string(index=False))

print('\nInterpretazione:')
print('   date → month   : ogni data appartiene a un solo mese')
print('   date → quarter : ogni data appartiene a un solo trimestre')
print('   date → year    : ogni data appartiene a un solo anno')


FD temporali scoperte: 3
   X       Y  card_X  card_Y
date   month    1688      56
date quarter    1688      19
date    year    1688       5

Interpretazione:
   date → month   : ogni data appartiene a un solo mese
   date → quarter : ogni data appartiene a un solo trimestre
   date → year    : ogni data appartiene a un solo anno


## 4. Attribute Tree (Approccio Data-Driven)

L'Attribute Tree viene costruito partendo dalla **fact root** ,
espandendo i rami secondo le FD scoperte e validate.

### Struttura dell'Attribute Tree per COVID-19:

```
Fact_Covid_Trend
    ├── iso_code ──→ location ──→ continent
    └── date ──→ month ──→ quarter ──→ year
```

- **Ramo geografico**: `iso_code → location → continent`
- **Ramo temporale**: `date → month → quarter → year`


In [10]:
# Classe che rappresenta l'Attribute Tree del DFM (radice, archi, misure) e lo disegna
class AttributeTree:
    """
    Rappresentazione dell'Attribute Tree per il DFM.

    Il nodo radice è il fatto (es. 'Fact_Covid_Trend').
    I rami rappresentano gerarchie derivate dalle FD.
    Le misure sono attributi del nodo radice.
    """

    def __init__(self, root):
        self.root     = root
        self.children = {root: []}  # nodo → figli
        self.parent   = {}          # nodo → padre
        self.measures = []          # misure associate al fatto
        self._nodes   = {root}

    def add_edge(self, parent, child):
        """Aggiunge un arco parent → child all'albero."""
        if parent not in self._nodes:
            self.children[parent] = []
            self._nodes.add(parent)
        if child not in self._nodes:
            self.children[child] = []
            self._nodes.add(child)
        if child not in self.children[parent]:
            self.children[parent].append(child)
        self.parent[child] = parent

    def add_measure(self, measure):
        """Aggiunge una misura al fatto."""
        if measure not in self.measures:
            self.measures.append(measure)

    def depth(self, node, d=0):
        """Calcola la profondità di un nodo dall'albero."""
        children = self.children.get(node, [])
        if not children:
            return d
        return max(self.depth(c, d+1) for c in children)

    def all_nodes(self):
        """Restituisce tutti i nodi dell'albero."""
        return list(self._nodes)

    def to_dict(self):
        """Serializza l'albero in dizionario (per schema.json)."""
        return {
            'root': self.root,
            'children': self.children,
            'measures': self.measures,
            'depth': self.depth(self.root),
            'nodes_count': len(self._nodes)
        }

    def visualize(self, figsize=(12, 6), title='Attribute Tree — COVID-19 DW'):
        """Disegna l'albero con networkx."""
        G = nx.DiGraph()

        def add_edges(node):
            for child in self.children.get(node, []):
                G.add_edge(node, child)
                add_edges(child)

        add_edges(self.root)
        for m in self.measures:
            G.add_edge(self.root, m)

        # Layout gerarchico
        pos = nx.drawing.nx_agraph.graphviz_layout(G, prog='dot') if               hasattr(nx.drawing, 'nx_agraph') else nx.spring_layout(G, seed=42)


        manual_pos = {
            self.root   : (0, 4),
            'iso_code'  : (-3, 2),
            'location'  : (-3, 1),
            'continent' : (-3, 0),
            'date'      : (3, 2),
            'month'     : (3, 1),
            'quarter'   : (3, 0),
            'year'      : (3, -1),
        }
        for m_i, m in enumerate(self.measures):
            manual_pos[m] = (0 + m_i*1.2 - len(self.measures)*0.6, 2.5)


        pos = {}
        for node in G.nodes():
            if node in manual_pos:
                pos[node] = manual_pos[node]
        missing = [n for n in G.nodes() if n not in pos]
        if missing:
            spring = nx.spring_layout(G, seed=42)
            for n in missing:
                pos[n] = spring[n]

        fig, ax = plt.subplots(figsize=figsize)

        # Colori nodi
        fact_nodes   = [self.root]
        dim_nodes    = [n for n in G.nodes() if n not in self.measures and n != self.root]
        meas_nodes   = self.measures

        nx.draw_networkx_nodes(G, pos, nodelist=fact_nodes,
                               node_color='#e74c3c', node_size=2500, ax=ax)
        nx.draw_networkx_nodes(G, pos, nodelist=dim_nodes,
                               node_color='#3498db', node_size=1800, ax=ax)
        nx.draw_networkx_nodes(G, pos, nodelist=meas_nodes,
                               node_color='#27ae60', node_size=1600, ax=ax)
        nx.draw_networkx_edges(G, pos, edge_color='#555', arrows=True,
                               arrowsize=20, ax=ax, connectionstyle='arc3,rad=0.05')
        nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

        # Legenda
        patches = [
            mpatches.Patch(color='#e74c3c', label='Fatto (root)'),
            mpatches.Patch(color='#3498db', label='Attributi / Dimensioni'),
            mpatches.Patch(color='#27ae60', label='Misure'),
        ]
        ax.legend(handles=patches, loc='lower right', fontsize=9)
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.axis('off')
        return fig, ax


print('Classe AttributeTree definita')

Classe AttributeTree definita


In [23]:
# Costruisco l'albero: radice = fatto, rami = gerarchie geografica e temporale, foglie = misure
# Costruzione dell'Attribute Tree COVID-19

tree = AttributeTree(root='Fact_Covid_Trend')

# Ramo geografico: iso_code → location → continent
tree.add_edge('Fact_Covid_Trend', 'iso_code')
tree.add_edge('iso_code',  'location')
tree.add_edge('location',  'continent')
# Validazione:
# • iso_code → location: ogni codice ISO identifica un paese univoco
# • location → continent: ogni paese appartiene a un solo continente

# Ramo temporale: date → month → quarter → year
tree.add_edge('Fact_Covid_Trend', 'date')
tree.add_edge('date',    'month')
tree.add_edge('month',   'quarter')
tree.add_edge('quarter', 'year')
# Validazione :
# • date → month   : ogni data ∈ un solo mese
# • month → quarter: ogni mese ∈ un solo trimestre
# • quarter → year : ogni trimestre ∈ un solo anno

# Misure
MEASURES = [
    'new_cases',
    'total_cases',
    'new_deaths',
    'total_deaths',
    'new_tests',
    'stringency_index',
    'people_vaccinated',
]
for m in MEASURES:
    tree.add_measure(m)

print('Attribute Tree costruito:')
print(f'   Radice:  {tree.root}')
print(f'   Nodi:    {len(tree.all_nodes())}')
print(f'   Misure:  {tree.measures}')
print(f'   Profondità: {tree.depth(tree.root)}')

Attribute Tree costruito:
   Radice:  Fact_Covid_Trend
   Nodi:    8
   Misure:  ['new_cases', 'total_cases', 'new_deaths', 'total_deaths', 'new_tests', 'stringency_index', 'people_vaccinated']
   Profondità: 4


## 5. Classificazione Additività delle Misure

La classificazione è fondamentale per garantire aggregazioni corrette nel DW:

| Tipo | Definizione | Aggregazione corretta | Esempi COVID |
|------|-------------|----------------------|--------------|
| **Additive** | Sommabile su tutte le dimensioni | SUM | `new_cases`, `new_deaths`, `new_tests` |
| **Semi-additive** | Sommabile su alcune dimensioni, non su time | MAX (nel tempo) | `total_cases`, `total_deaths`, `people_vaccinated` |
| **Non-additive** | Non sommabile — deve essere aggregata con AVG o ricalcolata | AVG | `stringency_index` |


> Il valore corretto è `MAX(total_cases)` ,perche sommare `(total_cases)` di più giorni è sbagliato, in quanto ogni giorno include anche i casi del giorno precedente. Cosi invece viene rappresentato il picco cumulativo.


In [26]:
# Classifico ogni misura per additivita (come si puo aggregare: SUM / MAX / AVG)
#Classificazione additività misure COVID

MEASURE_ADDITIVITY = {
    'new_cases':         {'additivity': 'additive',       'agg': 'SUM',
                          'reason': 'Nuovi casi giornalieri: sommabili su tutti gli assi'},
    'total_cases':       {'additivity': 'semi_additive',  'agg': 'MAX',
                          'reason': 'Cumulativo: sommare su time dà double-counting → MAX'},
    'new_deaths':        {'additivity': 'additive',       'agg': 'SUM',
                          'reason': 'Nuovi decessi giornalieri: sommabili su tutti gli assi'},
    'total_deaths':      {'additivity': 'semi_additive',  'agg': 'MAX',
                          'reason': 'Cumulativo: come total_cases → MAX'},
    'new_tests':         {'additivity': 'additive',       'agg': 'SUM',
                          'reason': 'Nuovi test giornalieri: sommabili'},
    'stringency_index':  {'additivity': 'non_additive',   'agg': 'AVG',
                          'reason': 'Indice 0-100: media ponderata — SUM non ha significato'},
    'people_vaccinated': {'additivity': 'semi_additive',  'agg': 'MAX',
                          'reason': 'Cumulativo: conta le persone vaccinate (stock) → MAX'},
}

print('Classificazione additività misure COVID-19:')
print()
print(f'  {"Misura":<22} {"Additività":<16} {"Agg.":>5}  Motivazione')
print('  ' + '─' * 80)

icons = {'additive': '', 'semi_additive': ' ', 'non_additive': ''}
for measure, info in MEASURE_ADDITIVITY.items():
    icon = icons[info['additivity']]
    print(f'  {icon} {measure:<20} {info["additivity"]:<16} {info["agg"]:>5}  {info["reason"]}')

Classificazione additività misure COVID-19:

  Misura                 Additività        Agg.  Motivazione
  ────────────────────────────────────────────────────────────────────────────────
   new_cases            additive           SUM  Nuovi casi giornalieri: sommabili su tutti gli assi
    total_cases          semi_additive      MAX  Cumulativo: sommare su time dà double-counting → MAX
   new_deaths           additive           SUM  Nuovi decessi giornalieri: sommabili su tutti gli assi
    total_deaths         semi_additive      MAX  Cumulativo: come total_cases → MAX
   new_tests            additive           SUM  Nuovi test giornalieri: sommabili
   stringency_index     non_additive       AVG  Indice 0-100: media ponderata — SUM non ha significato
    people_vaccinated    semi_additive      MAX  Cumulativo: conta le persone vaccinate (stock) → MAX


##  6. Star Schema

Traduzione del DFM nel modello logico a stella:

```
Fact_Covid_Trend
    ├── FK: dim_location_context_sk  →  Dim_Location_Context (iso_code, location, continent)
    └── FK: dim_time_sk              →  Dim_Time              (date, month, quarter, year)
```

- **Fact_Covid_Trend**: una riga per (iso_code, date) — le misure epidemiologiche
- **Dim_Location_Context**: una riga per paese — attributi geografici e demografici
- **Dim_Time**: una riga per data — gerarchia temporale


In [27]:
# Funzione che traduce l'Attribute Tree nello star schema logico (fatto + dimensioni)
def build_star_schema(tree):
    """
    Traduce l'Attribute Tree nel logical Star Schema.
    Produce il dizionario usato per DDL e schema.json.
    """
    star = {
        'fact_table': {
            'name': tree.root,
            'grain': 'Una riga per (iso_code, date): misure epidemiologiche giornaliere per paese',
            'measures': tree.measures,
            'foreign_keys': []
        },
        'dimension_tables': {}
    }

    for dim_child in tree.children.get(tree.root, []):
        # Raccolgo tutti i livelli gerarchici in BFS
        levels = [dim_child]
        stack = list(tree.children.get(dim_child, []))
        while stack:
            n = stack.pop(0)
            levels.append(n)
            stack.extend(tree.children.get(n, []))

        # Nome tabella dimensionale
        dim_name_map = {
            'iso_code': 'Dim_Location_Context',
            'date':     'Dim_Time'
        }
        dim_name = dim_name_map.get(dim_child, f'Dim_{dim_child.title()}')

        # Chiave surrogata
        sk_map = {
            'Dim_Location_Context': 'location_sk',
            'Dim_Time':             'time_sk'
        }
        sk = sk_map.get(dim_name, f'{dim_name.lower()}_sk')

        star['dimension_tables'][dim_name] = {
            'surrogate_key': sk,
            'natural_key':   dim_child,
            'levels':        levels,
            'description':   'Gerarchia: ' + ' → '.join(levels)
        }
        star['fact_table']['foreign_keys'].append(sk)

    return star


star_schema = build_star_schema(tree)

print(' STAR SCHEMA — COVID-19 Data Warehouse')
print('═' * 55)
print(f'\n   FATTO: {star_schema["fact_table"]["name"]}')
print(f'     Grain: {star_schema["fact_table"]["grain"]}')
print(f'     Misure: {star_schema["fact_table"]["measures"]}')
print(f'     FK:     {star_schema["fact_table"]["foreign_keys"]}')
print()
for dim_name, meta in star_schema['dimension_tables'].items():
    print(f'    DIMENSIONE: {dim_name}')
    print(f'     Surrogate key: {meta["surrogate_key"]}')
    print(f'     Natural key:   {meta["natural_key"]}')
    print(f'     {meta["description"]}')
    print()

 STAR SCHEMA — COVID-19 Data Warehouse
═══════════════════════════════════════════════════════

   FATTO: Fact_Covid_Trend
     Grain: Una riga per (iso_code, date): misure epidemiologiche giornaliere per paese
     Misure: ['new_cases', 'total_cases', 'new_deaths', 'total_deaths', 'new_tests', 'stringency_index', 'people_vaccinated']
     FK:     ['location_sk', 'time_sk']

    DIMENSIONE: Dim_Location_Context
     Surrogate key: location_sk
     Natural key:   iso_code
     Gerarchia: iso_code → location → continent

    DIMENSIONE: Dim_Time
     Surrogate key: time_sk
     Natural key:   date
     Gerarchia: date → month → quarter → year



## 7. Generazione DDL MySQL

Generiamo gli statement `CREATE TABLE` per il DW MySQL partendo dallo Star Schema.
Le dimensioni vengono create prima del fatto (vincoli FK).


In [28]:
# Funzione che genera il DDL MySQL (CREATE TABLE) a partire dallo star schema
def generate_ddl_mysql(star, measure_additivity):
    """
    Genera DDL MySQL per la Star Schema del DW COVID-19.
    Crea prima le tabelle dimensionali, poi la fact table.
    Include commenti con la classificazione di additività.
    """
    ddl = []
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # Header
    ddl.append('-- ═══════════════════════════════════════════════════════════════')
    ddl.append('-- DATA WAREHOUSE — COVID-19 Trend Analysis')
    ddl.append('-- Studente: Costantino Chiara | Matricola: 277081 | UNICAL')
    ddl.append(f'-- Generato: {ts}')
    ddl.append('-- ═══════════════════════════════════════════════════════════════')
    ddl.append('')
    ddl.append('CREATE DATABASE IF NOT EXISTS covid_dw CHARACTER SET utf8mb4;')
    ddl.append('USE covid_dw;')
    ddl.append('')

    #Colonne aggiuntive per le dimensioni
    dim_extra_cols = {
        'Dim_Location_Context': [
            '    population              BIGINT          COMMENT "Popolazione residente"',
            '    population_density      FLOAT           COMMENT "Abitanti per km²"',
            '    gdp_per_capita          FLOAT           COMMENT "PIL pro capite (USD)"',
            '    median_age              FLOAT           COMMENT "Età mediana della popolazione"',
            '    hospital_beds_per_thousand FLOAT        COMMENT "Posti letto ospedalieri per 1000 ab."',
            '    human_development_index FLOAT           COMMENT "Indice di sviluppo umano (0-1)"',
        ],
        'Dim_Time': [
            '    day_of_week                 TINYINT         COMMENT "Giorno della settimana (0=lunedi)"',
            '    is_weekend                  BOOLEAN         COMMENT "TRUE se sabato o domenica"',
            '    days_since_start            INT             COMMENT "Giorni dal 2020-01-01"',
        ]
    }

    # Tabelle dimensionali
    for dim_name, meta in star['dimension_tables'].items():
        sk   = meta['surrogate_key']
        nk   = meta['natural_key']
        lvls = meta['levels']

        ddl.append(f'-- ─── {dim_name} ──────────────────────────────────────────')
        ddl.append(f'CREATE TABLE IF NOT EXISTS {dim_name} (')

        cols = []
        cols.append(f'    {sk:<35} INT AUTO_INCREMENT PRIMARY KEY COMMENT "Surrogate key"')
        cols.append(f'    {nk:<35} VARCHAR(10)  NOT NULL UNIQUE   COMMENT "Natural key (business key)"')

        # Livelli gerarchici
        type_map = {
            'location': 'VARCHAR(100)', 'continent': 'VARCHAR(50)',
            'month': 'VARCHAR(7)', 'quarter': 'VARCHAR(7)', 'year': 'SMALLINT'
        }
        for lvl in lvls[1:]:
            col_type = type_map.get(lvl, 'VARCHAR(100)')
            cols.append(f'    {lvl:<35} {col_type:<14} COMMENT "Livello gerarchico"')

        # Colonne aggiuntive specifiche
        for extra in dim_extra_cols.get(dim_name, []):
            if extra:
                cols.append(extra)

        # Timestamp ETL
        cols.append('    load_timestamp              TIMESTAMP       DEFAULT CURRENT_TIMESTAMP COMMENT "ETL timestamp"')

        ddl.append(',\n'.join(cols))
        ddl.append(') ENGINE=InnoDB CHARACTER SET utf8mb4;')
        ddl.append('')

    # Fact table
    fact = star['fact_table']
    fks  = star['dimension_tables']

    ddl.append(f'-- ─── {fact["name"]} ─────────────────────────────────────────────')
    ddl.append(f'CREATE TABLE IF NOT EXISTS {fact["name"]} (')

    cols = []
    cols.append(f'    {"fact_sk":<35} BIGINT AUTO_INCREMENT PRIMARY KEY COMMENT "Surrogate key fatto"')

    # Foreign keys
    for dim_name, meta in fks.items():
        sk = meta['surrogate_key']
        cols.append(f'    {sk:<35} INT NOT NULL  COMMENT "FK → {dim_name}"')

    # Misure con classificazione additività
    measure_type_map = {
        'new_cases':         'FLOAT',
        'total_cases':       'FLOAT',
        'new_deaths':        'FLOAT',
        'total_deaths':      'FLOAT',
        'new_tests':         'FLOAT',
        'stringency_index':  'FLOAT',
        'people_vaccinated': 'BIGINT',
    }
    for m in fact['measures']:
        col_type = measure_type_map.get(m, 'FLOAT')
        info     = measure_additivity.get(m, {})
        add_type = info.get('additivity', 'additive')
        agg      = info.get('agg', 'SUM')
        cols.append(f'    {m:<35} {col_type:<8} COMMENT "{add_type} [{agg}]"')


    # Timestamp ETL
    cols.append('    load_timestamp                      TIMESTAMP DEFAULT CURRENT_TIMESTAMP COMMENT "ETL timestamp"')

    # Vincoli: grain (UNIQUE KEY) + FK verso le dimensioni
    fk_constraints = []
    fk_constraints.append('    UNIQUE KEY uk_grain (location_sk, time_sk)')
    for dim_name, meta in fks.items():
        sk = meta['surrogate_key']
        fk_constraints.append(f'    CONSTRAINT fk_{sk} FOREIGN KEY ({sk}) REFERENCES {dim_name}({sk})')

    ddl.append(',\n'.join(cols + fk_constraints))
    ddl.append(') ENGINE=InnoDB CHARACTER SET utf8mb4;')
    ddl.append('')

    # Indici per performance query
    ddl.append('-- ─── Indici per performance ─────────────────────────────────────────')
    ddl.append("-- Nota: l'indice composto (location_sk, time_sk) e' gia' fornito da uk_grain")
    ddl.append(f'CREATE INDEX idx_fact_loc ON {fact["name"]}(location_sk);')
    ddl.append(f'CREATE INDEX idx_fact_time ON {fact["name"]}(time_sk);')
    ddl.append('')

    return '\n'.join(ddl)


ddl_sql = generate_ddl_mysql(star_schema, MEASURE_ADDITIVITY)

# Salva DDL
with open(OUTPUT_DDL, 'w', encoding='utf-8') as f:
    f.write(ddl_sql)

print(' DDL MySQL generato e salvato:')
print(f'   {OUTPUT_DDL}')
print()
print('─' * 65)
print(ddl_sql[:3000])
print('...')

 DDL MySQL generato e salvato:
   /content/drive/MyDrive/DataWarehouse/schema_dw.sql

─────────────────────────────────────────────────────────────────
-- ═══════════════════════════════════════════════════════════════
-- DATA WAREHOUSE — COVID-19 Trend Analysis
-- Studente: Costantino Chiara | Matricola: 277081 | UNICAL
-- Generato: 2026-06-14 08:00:31
-- ═══════════════════════════════════════════════════════════════

CREATE DATABASE IF NOT EXISTS covid_dw CHARACTER SET utf8mb4;
USE covid_dw;

-- ─── Dim_Location_Context ──────────────────────────────────────────
CREATE TABLE IF NOT EXISTS Dim_Location_Context (
    location_sk                         INT AUTO_INCREMENT PRIMARY KEY COMMENT "Surrogate key",
    iso_code                            VARCHAR(10)  NOT NULL UNIQUE   COMMENT "Natural key (business key)",
    location                            VARCHAR(100)   COMMENT "Livello gerarchico",
    continent                           VARCHAR(50)    COMMENT "Livello gerarchico",
   

## 8. Costruzione Fisica delle Tabelle DW

Costruiamo le tabelle `Dim_Time`, `Dim_Location_Context` e `Fact_Covid_Trend`
come DataFrame pandas, pronti per il caricamento nel DBMS tramite ETL.


In [29]:
# Costruisco la dimensione tempo (Dim_Time): una riga per ogni giorno del periodo
# Dim_Time
all_dates = pd.date_range(
    start=epidemic_df['date'].min(),
    end=epidemic_df['date'].max(),
    freq='D'
)

start_date = pd.Timestamp('2020-01-01')
dim_time = pd.DataFrame({
    'time_sk':          range(1, len(all_dates) + 1),
    'date':             all_dates,
    'month':            all_dates.to_period('M').astype(str),
    'quarter':          all_dates.to_period('Q').astype(str),
    'year':             all_dates.year,
    'day_of_week':      all_dates.dayofweek,
    'is_weekend':       all_dates.dayofweek >= 5,
    'days_since_start': (all_dates - start_date).days
})

dim_time.to_csv(OUTPUT_DIM_TIME, index=False)
print(f'Dim_Time: {len(dim_time):,} righe → {OUTPUT_DIM_TIME}')
dim_time.head(3)

Dim_Time: 1,688 righe → /content/drive/MyDrive/DataWarehouse/Dim_Time.csv


,time_sk,date,month,quarter,year,day_of_week,is_weekend,days_since_start
0,1,2020-01-01,2020-01,2020Q1,2020,2,False,0
1,2,2020-01-02,2020-01,2020Q1,2020,3,False,1
2,3,2020-01-03,2020-01,2020Q1,2020,4,False,2


In [17]:
# Costruisco la dimensione luogo (Dim_Location_Context) con la sua chiave surrogata
# Dim_Location_Context
LOC_DIM_COLS = [
    'iso_code', 'location', 'continent',
    'population', 'population_density', 'gdp_per_capita',
    'median_age', 'hospital_beds_per_thousand', 'human_development_index'
]
available_loc_cols = [c for c in LOC_DIM_COLS if c in locations_df.columns]

dim_location = (locations_df[available_loc_cols]
                .dropna(subset=['iso_code'])
                .drop_duplicates(subset=['iso_code'])
                .reset_index(drop=True))

dim_location.insert(0, 'location_sk', range(1, len(dim_location) + 1))

dim_location.to_csv(OUTPUT_DIM_LOC, index=False)
print(f' Dim_Location_Context: {len(dim_location):,} righe → {OUTPUT_DIM_LOC}')
dim_location.head(3)

 Dim_Location_Context: 171 righe → /content/drive/MyDrive/DataWarehouse/Dim_Location_Context.csv


,location_sk,iso_code,location,continent,population,population_density,gdp_per_capita,median_age,hospital_beds_per_thousand,human_development_index
0,1,AFG,Afghanistan,Asia,41128772.0,54.42,1803.99,18.6,0.50,0.51
1,2,ALB,Albania,Europe,2842318.0,104.87,11803.43,38.0,2.89,0.80
2,3,DZA,Algeria,Africa,44903228.0,17.35,13913.84,29.1,1.90,0.75


In [18]:
# Costruisco la fact table: unisco le misure e collego le chiavi surrogate delle dimensioni
# Fact_Covid_Trend
# Unisco epidemic + prevention per paese-data
fact_raw = epidemic_df.merge(
    prevention_df[['iso_code', 'date', 'stringency_index', 'people_vaccinated']],
    on=['iso_code', 'date'],
    how='left'
)

# Aggiungo surrogate keys
loc_sk_map  = dim_location.set_index('iso_code')['location_sk'].to_dict()
time_sk_map = dim_time.set_index('date')['time_sk'].to_dict()

# Converto la colonna date a Timestamp se necessario
fact_raw['date'] = pd.to_datetime(fact_raw['date'])

fact_raw['location_sk'] = fact_raw['iso_code'].map(loc_sk_map)
fact_raw['time_sk']     = fact_raw['date'].map(time_sk_map)

# Seleziono solo le colonne del fatto
FACT_COLS = (
    ['fact_sk', 'location_sk', 'time_sk'] +
    [m for m in MEASURE_ADDITIVITY.keys() if m in fact_raw.columns]
)

# Filtro righe con SK valide
fact_df = fact_raw.dropna(subset=['location_sk', 'time_sk']).copy()
fact_df['location_sk'] = fact_df['location_sk'].astype(int)
fact_df['time_sk']     = fact_df['time_sk'].astype(int)
fact_df.insert(0, 'fact_sk', range(1, len(fact_df) + 1))

available_fact_cols = [c for c in FACT_COLS if c in fact_df.columns]
fact_df = fact_df[available_fact_cols]

fact_df.to_csv(OUTPUT_FACT, index=False)
print(f' Fact_Covid_Trend: {len(fact_df):,} righe → {OUTPUT_FACT}')
print(f'   Colonne: {list(fact_df.columns)}')
fact_df.head(3)

 Fact_Covid_Trend: 284,298 righe → /content/drive/MyDrive/DataWarehouse/Fact_Covid_Trend.csv
   Colonne: ['fact_sk', 'location_sk', 'time_sk', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths', 'stringency_index', 'people_vaccinated']


,fact_sk,location_sk,time_sk,new_cases,total_cases,new_deaths,total_deaths,stringency_index,people_vaccinated
0,1,1,5,0.0,0.0,0.0,0.0,0.0,0.0
1,2,1,6,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1,7,0.0,0.0,0.0,0.0,0.0,0.0


## 9. Glossario delle Misure




In [30]:
# Glossario delle misure: definizione, fonte, additivita e note per ognuna
# Glossario delle misure COVID-19
glossary = [
    {
        'measure':            'new_cases',
        'definition':         'Nuovi casi COVID-19 confermati nella giornata',
        'unit':               'numero di casi (conteggio)',
        'source':             'OWID → colonna new_cases',
        'computation':        'Segnalazione giornaliera; NaN → 0 (zero-imputation MCAR)',
        'additivity':         'additive',
        'valid_aggregations': 'SUM, AVG, MIN, MAX',
        'notes':              'I valori negativi (correzioni OWID retroattive) sono posti a 0'
    },
    {
        'measure':            'total_cases',
        'definition':         'Totale cumulativo di casi COVID-19 confermati dallo inizio della pandemia',
        'unit':               'numero di casi (conteggio cumulativo)',
        'source':             'OWID → colonna total_cases',
        'computation':        'Cumulativo progressivo; non imputato (NaN = dato non ancora disponibile)',
        'additivity':         'semi_additive',
        'valid_aggregations': 'MAX (nel tempo), SUM (tra paesi in un singolo giorno)',
        'notes':              'NON sommare su più giorni: produce double-counting'
    },
    {
        'measure':            'new_deaths',
        'definition':         'Nuovi decessi COVID-19 nella giornata',
        'unit':               'numero di decessi (conteggio)',
        'source':             'OWID → colonna new_deaths',
        'computation':        'Segnalazione giornaliera; NaN → 0 (zero-imputation MCAR)',
        'additivity':         'additive',
        'valid_aggregations': 'SUM, AVG, MIN, MAX',
        'notes':              'I picchi possono riflettere conteggi in ritardo segnalati in un solo giorno'
    },
    {
        'measure':            'total_deaths',
        'definition':         'Totale cumulativo di decessi COVID-19',
        'unit':               'numero di decessi (conteggio cumulativo)',
        'source':             'OWID → colonna total_deaths',
        'computation':        'Cumulativo progressivo',
        'additivity':         'semi_additive',
        'valid_aggregations': 'MAX (nel tempo), SUM (tra paesi in un singolo giorno)',
        'notes':              'Stessa logica di total_cases — NON sommare su time'
    },
    {
        'measure':            'new_tests',
        'definition':         'Nuovi test diagnostici COVID-19 eseguiti nella giornata',
        'unit':               'numero di test (conteggio)',
        'source':             'OWID → colonna new_tests',
        'computation':        'Segnalazione giornaliera; NaN → 0',
        'additivity':         'additive',
        'valid_aggregations': 'SUM, AVG',
        'notes':              'Disponibile solo per alcuni paesi; assenza = NaN, non zero'
    },
    {
        'measure':            'stringency_index',
        'definition':         'Indice di rigidità delle politiche di contenimento COVID (scala 0-100)',
        'unit':               'indice adimensionale [0, 100]',
        'source':             'OWID → colonna stringency_index (da Oxford COVID-19 Government Response Tracker)',
        'computation':        'Media ponderata di 9 indicatori di policy (school closures, workplace closures, ecc.)',
        'additivity':         'non_additive',
        'valid_aggregations': 'AVG, MIN, MAX',
        'notes':              'NON sommare: SUM non ha significato; usare AVG per aggregazioni temporali e geografiche'
    },
    {
        'measure':            'people_vaccinated',
        'definition':         'Numero cumulativo di persone che hanno ricevuto almeno una dose di vaccino COVID-19',
        'unit':               'numero di persone (conteggio cumulativo)',
        'source':             'OWID → colonna people_vaccinated',
        'computation':        'Cumulativo; forward-fill per paese (MAR); NaN pre-2021 → 0 (MNAR)',
        'additivity':         'semi_additive',
        'valid_aggregations': 'MAX (nel tempo), SUM (tra paesi in un singolo giorno per stima globale)',
        'notes':              'La campagna vaccinale è iniziata nella maggior parte dei paesi nel 2021; pre-2021 = 0'
    },
]

glossary_df = pd.DataFrame(glossary)


print(' GLOSSARIO DELLE MISURE — COVID-19 Data Warehouse')
print('=' * 70)
print()
for g in glossary:
    print(f'━━━ {g["measure"].upper()} ━━━')
    print(f'  Definizione:    {g["definition"]}')
    print(f'  Unità:          {g["unit"]}')
    print(f'  Fonte:          {g["source"]}')
    print(f'  Calcolo:        {g["computation"]}')
    print(f'  Additività:     {g["additivity"]}')
    print(f'  Aggregazioni:   {g["valid_aggregations"]}')
    print(f'  Note:           {g["notes"]}')
    print()

glossary_df.to_csv(OUTPUT_GLOSSARY, index=False)
print(f' Glossario salvato → {OUTPUT_GLOSSARY}')

 GLOSSARIO DELLE MISURE — COVID-19 Data Warehouse

━━━ NEW_CASES ━━━
  Definizione:    Nuovi casi COVID-19 confermati nella giornata
  Unità:          numero di casi (conteggio)
  Fonte:          OWID → colonna new_cases
  Calcolo:        Segnalazione giornaliera; NaN → 0 (zero-imputation MCAR)
  Additività:     additive
  Aggregazioni:   SUM, AVG, MIN, MAX
  Note:           I valori negativi (correzioni OWID retroattive) sono posti a 0

━━━ TOTAL_CASES ━━━
  Definizione:    Totale cumulativo di casi COVID-19 confermati dallo inizio della pandemia
  Unità:          numero di casi (conteggio cumulativo)
  Fonte:          OWID → colonna total_cases
  Calcolo:        Cumulativo progressivo; non imputato (NaN = dato non ancora disponibile)
  Additività:     semi_additive
  Aggregazioni:   MAX (nel tempo), SUM (tra paesi in un singolo giorno)
  Note:           NON sommare su più giorni: produce double-counting

━━━ NEW_DEATHS ━━━
  Definizione:    Nuovi decessi COVID-19 nella giornata
  Uni

## 10. Export `schema.json` per ETL

Esportiamo il contratto tra L4 (Modeling) e il passo ETL successivo.
Il file `schema.json` contiene tutte le informazioni necessarie al processo ETL:
Attribute Tree, Star Schema, misure, DDL path e glossario.


In [20]:
# Esporto schema.json: il "contratto" che descrive il DW per il notebook ETL successivo
#  Costruzione schema.json

schema_export = {
    'generated_at':  datetime.now().isoformat(),
    'source':        'COVID_L4_Data_Modeling.ipynb',
    'project':       'COVID-19 Trend Analysis — Data Warehouse',
    'student':       'Costantino Chiara | Matricola 277081 | UNICAL',

    'reconciled_db': {
        'tables': ['Locations', 'Epidemic_Trend', 'Prevention_Measures'],
        'location': RECONCILED_PATH
    },

    'attribute_tree': tree.to_dict(),

    'measures': {
        name: {
            'additivity': info['additivity'],
            'aggregation': info['agg'],
            'reason': info['reason']
        }
        for name, info in MEASURE_ADDITIVITY.items()
    },

    'star_schema': star_schema,
    'glossary': glossary,
    'ddl_file': OUTPUT_DDL,

    'dimension_tables': {
        'Dim_Time': {
            'path': OUTPUT_DIM_TIME,
            'rows': len(dim_time),
            'surrogate_key': 'time_sk',
            'natural_key': 'date'
        },
        'Dim_Location_Context': {
            'path': OUTPUT_DIM_LOC,
            'rows': len(dim_location),
            'surrogate_key': 'location_sk',
            'natural_key': 'iso_code'
        },
    },

    'fact_table': {
        'name': 'Fact_Covid_Trend',
        'path': OUTPUT_FACT,
        'rows': len(fact_df),
        'grain': 'Una riga per (iso_code, date)',
        'foreign_keys': ['location_sk', 'time_sk']
    }
}

with open(OUTPUT_SCHEMA, 'w', encoding='utf-8') as f:
    json.dump(schema_export, f, indent=2, ensure_ascii=False, default=str)

print(' schema.json esportato')
print()
print(' File prodotti da questo notebook:')
outputs = [
    (OUTPUT_DIM_TIME,   f'Dim_Time ({len(dim_time):,} righe)'),
    (OUTPUT_DIM_LOC,    f'Dim_Location_Context ({len(dim_location):,} righe)'),
    (OUTPUT_FACT,       f'Fact_Covid_Trend ({len(fact_df):,} righe)'),
    (OUTPUT_DDL,        'DDL MySQL (CREATE TABLE statements)'),
    (OUTPUT_GLOSSARY,   'Glossario misure (CSV)'),
    (OUTPUT_SCHEMA,     'schema.json (contratto per ETL)'),
    (BASE_PATH + 'L4_attribute_tree.png', 'Visualizzazione Attribute Tree'),
]
for path, desc in outputs:
    print(f'   ✓ {desc}')
    print(f'     → {path}')

 schema.json esportato

 File prodotti da questo notebook:
   ✓ Dim_Time (1,688 righe)
     → /content/drive/MyDrive/DataWarehouse/Dim_Time.csv
   ✓ Dim_Location_Context (171 righe)
     → /content/drive/MyDrive/DataWarehouse/Dim_Location_Context.csv
   ✓ Fact_Covid_Trend (284,298 righe)
     → /content/drive/MyDrive/DataWarehouse/Fact_Covid_Trend.csv
   ✓ DDL MySQL (CREATE TABLE statements)
     → /content/drive/MyDrive/DataWarehouse/schema_dw.sql
   ✓ Glossario misure (CSV)
     → /content/drive/MyDrive/DataWarehouse/glossary_measures.csv
   ✓ schema.json (contratto per ETL)
     → /content/drive/MyDrive/DataWarehouse/schema.json
   ✓ Visualizzazione Attribute Tree
     → /content/drive/MyDrive/DataWarehouse/L4_attribute_tree.png


## 🎯 11. Riepilogo

✅ Reconciled DB caricato
   Locations (paesi) · Epidemic_Trend (flussi) · Prevention_Measures (vaccinazioni)

✅ Dipendenze Funzionali scoperte e validate:
  

✅ Attribute Tree (approccio data-driven):
   Fact_Covid_Trend
       ├── iso_code → location → continent
       └── date → month → quarter → year

✅ Misure classificate per additività:
   • new_cases, new_deaths, new_tests         → additive      [SUM]
   • total_cases, total_deaths, people_vacc.  → semi-additive [MAX]
   • stringency_index                         → non-additive  [AVG]

✅ DFM Fact Schema documentato

✅ Star Schema logico:
   Fact_Covid_Trend + Dim_Location_Context + Dim_Time

✅ Tabelle DW costruite fisicamente come DataFrame:
   Dim_Time · Dim_Location_Context · Fact_Covid_Trend

✅ DDL MySQL generato (schema_dw.sql)

✅ Glossario delle misure

✅ schema.json esportato


### Principi DW applicati:
- **Surrogate key**: integer auto-increment per ogni dimensione
- **Natural key preservato**: tracciabilità verso il dato sorgente
- **Denormalizzazione**: gerarchie appiattite nella dimensione (Star ≠ Snowflake)
- **Additività documentata**: ogni misura ha la regola di aggregazione corretta
- **Audit trail**: ogni step di trasformazione è documentato (L3)

 **Prossimo step (L5):** ETL pipeline dal Reconciled DB al DW
